In [0]:
%run "../00.common/01.environment-config.py"

In [0]:
# COMMAND ----------
 
# Load environment configuration
 
# MAGIC %run "../00_common/01_environment_config"
 
 
# COMMAND ----------
 
# ============================================================
# SOURCE AND TARGET PATHS
# ============================================================
 
bronze_path = f"{s3_root_path}/Bronze/Nisarga/circuits"
 
silver_path = f"{s3_root_path}/Silver/Nisarga/circuits"
 
print("Bronze Path :", bronze_path)
print("Silver Path :", silver_path)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 1 - READ BRONZE CIRCUITS DATA
# ============================================================
 
circuits_df = (
    spark.read
    .format("delta")
    .load(bronze_path)
)
 
display(circuits_df)
 
 
# COMMAND ----------
 
# Import PySpark functions
 
from pyspark.sql import functions as F
 
 
# COMMAND ----------
 
# ============================================================
# STEP 2 - SELECT REQUIRED COLUMNS
# ============================================================
# URL is not required for analytics, so it is excluded.
 
circuits_selected_df = circuits_df.select(
    F.col("circuitId"),
    F.col("circuitName"),
    F.col("lat"),
    F.col("long"),
    F.col("locality"),
    F.col("country"),
    F.col("ingestion_timestamp"),
    F.col("source_file")
)
 
display(circuits_selected_df)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 3 - STANDARDIZE COLUMN NAMES
# ============================================================
 
circuits_renamed_df = (
    circuits_selected_df
    .withColumnsRenamed({
        "circuitId": "circuit_id",
        "circuitName": "circuit_name",
        "lat": "latitude",
        "long": "longitude"
    })
)
 
display(circuits_renamed_df)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 4 - REMOVE NULL BUSINESS KEYS
# ============================================================
# circuit_id is the business key, so records without a
# circuit_id are not kept in Silver.
 
circuits_valid_df = (
    circuits_renamed_df
    .filter(F.col("circuit_id").isNotNull())
)
 
display(circuits_valid_df)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 5 - REMOVE DUPLICATES
# ============================================================
 
circuits_distinct_df = (
    circuits_valid_df
    .dropDuplicates(["circuit_id"])
)
 
display(circuits_distinct_df)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 6 - STANDARDIZE TEXT VALUES
# ============================================================
 
circuits_final_df = (
    circuits_distinct_df
    .withColumn(
        "circuit_name",
        F.initcap(F.col("circuit_name"))
    )
    .withColumn(
        "locality",
        F.initcap(F.col("locality"))
    )
)
 
display(circuits_final_df)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 7 - WRITE TO SILVER IN DELTA FORMAT
# ============================================================
 
(
    circuits_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(silver_path)
)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 8 - VALIDATE SILVER DATA
# ============================================================
 
silver_circuits_df = (
    spark.read
    .format("delta")
    .load(silver_path)
)
 
display(silver_circuits_df)
 
print("Bronze Records :", circuits_df.count())
print("Silver Records :", silver_circuits_df.count())
 
print("Silver Circuits data successfully written to:")
print(silver_path)
 